In [416]:
import torch
import torch.nn as nn
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import math
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [417]:
import pickle
with open('weights/word2index_eng.pkl', 'rb') as f:
    word2index_eng = pickle.load(f)
with open('weights/word2index_fr.pkl', 'rb') as f:
    word2index_fr = pickle.load(f)

index2word_eng = {id:word for (word, id) in word2index_eng.items()}
index2word_fr = {id:word for (word, id) in word2index_fr.items()}

nlp_eng = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

In [418]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, nlp_eng, nlp_fr, word2index_eng, word2index_fr, root_eng, root_fr):
        self.data = []
        doc_bin_eng = DocBin().from_disk(root_eng).get_docs(nlp_eng.vocab)
        doc_bin_fr = DocBin().from_disk(root_fr).get_docs(nlp_fr.vocab)
        i = 0
        for doc_eng, doc_fr in zip(doc_bin_eng, doc_bin_fr):
            i += 1
            if i > 240500:
                continue
            sentence_eng = [2]
            for token in doc_eng:
                if not token.is_punct and not token.is_space:
                    sentence_eng.append(word2index_eng[token.text.lower()])
            
            sentence_fr = [2]
            for token in doc_fr:
                if not token.is_punct and not token.is_space:
                    sentence_fr.append(word2index_fr[token.text.lower()])

            sentence_eng.append(3)
            sentence_fr.append(3)

            sentence_eng = self.padding(sentence_eng, 40)
            sentence_fr = self.padding(sentence_fr, 40)

            self.data.append((torch.tensor(sentence_eng), torch.tensor(sentence_fr)))
    
    def padding(self, vector, size):
        while len(vector) != size:
            vector.append(0)
        return vector
    
    def __getitem__(self, index):
        return self.data[index]
    def __len__(self):
        return len(self.data)

In [ ]:
dataset = CustomDataset(nlp_eng, nlp_fr, word2index_eng, word2index_fr, 'eng_data.spacy', 'fr_data.spacy')
dataloader = torch.utils.data.DataLoader(dataset, 32, True)
len(dataset)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 1. Create a matrix of zeros: shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        
        # 2. Create a column vector of positions: 0, 1, 2, 3... shape (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
         
        # 3. Calculate the denominator (we use exponential and log for numerical stability)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # 4. Apply Sine to the even indices (0, 2, 4...) of the d_model dimension
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # 5. Apply Cosine to the odd indices (1, 3, 5...) of the d_model dimension
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 6. Add a batch dimension at the front: shape (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # CRITICAL: Register as a buffer. 
        # This tells PyTorch "save this to the GPU, but DO NOT calculate gradients for it."
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (Batch, Sequence_Length, d_model)
        
        # Slice the positional encoding to perfectly match the sequence length of x, then add them
        x = x + self.pe[:, :x.size(1), :]
        
        return self.dropout(x)

In [ ]:
class CustomMultiheadAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.d = d
        self.heads = heads
        
        self.kL = nn.Linear(
            in_features = d,
            out_features = d
        )
        self.qL = nn.Linear(
            in_features = d,
            out_features = d
        )
        self.vL = nn.Linear(
            in_features = d,
            out_features = d
        )
        self.out = nn.Linear(
            in_features = d,
            out_features = d
        )

    def forward(self, x: torch.Tensor, mask = None, decoder = False):
        batch_size = x.shape[0]
        num_words = x.shape[1]
        K: torch.Tensor = self.kL(x)
        Q = self.kL(x)
        V = self.kL(x)
        #batch_size x num_words x d ----> batch_size x num_words x heads x (d//heads) ----> batch_size x heads x num_words x (d//heads)
        K = K.view(batch_size, num_words, self.heads, -1)
        Q = Q.view(batch_size, num_words, self.heads, -1)
        V = V.view(batch_size, num_words, self.heads, -1)

        K = K.transpose(1, 2)
        Q = Q.transpose(1, 2)
        V = V.transpose(1, 2)

        energy = torch.matmul(Q, K.transpose(-1, -2))
        energy /= math.sqrt(self.d)
        if mask is not None:
            if not decoder:
                energy = energy.masked_fill(mask==0, float("-1e20"))
            else:
                trg_len = x.shape[1]
                trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=DEVICE)).bool()
                mask = mask & trg_sub_mask
                energy = energy.masked_fill(mask==0, float("-1e20"))
                
        attn_weights = torch.softmax(energy, -1)
        
        outs = torch.matmul(attn_weights, V)
        outs = outs.transpose(1, 2).contiguous()
        outs = outs.view(batch_size, num_words, -1)
        return self.out(outs)


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.mha = CustomMultiheadAttention(d, heads)
        self.ff = nn.Sequential(
            nn.Linear(
                in_features = d,
                out_features = d*2
            ),
            nn.Tanh(),
            nn.Linear(
                in_features = d*2,
                out_features = d
            )
        )
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x, mask = None):
        attn = self.mha(x, mask)
        x = self.norm1(x+attn)
        
        after_ff = self.ff(x)
        x = self.norm2(x+after_ff)

        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d, heads, num_layers, w2v_eng_root, max_len):
        super().__init__()
        self.d = d
        self.heads = heads
        self.layer_stack = nn.Sequential(*[EncoderBlock(d, heads) for _ in range(num_layers-1)])
        self.embedding = nn.Embedding.from_pretrained(torch.load(w2v_eng_root, weights_only=True), freeze=True).to(DEVICE)
        self.pos_embedding = PositionalEncoding(d, max_len).to(DEVICE)
        self.pad_idx = 0
        self.block1 = EncoderBlock(self.d, self.heads).to(DEVICE)
    def forward(self, x):
        src_mask = (x != self.pad_idx).unsqueeze(1).unsqueeze(2).to(DEVICE)
        out = self.embedding(x)
        out *= math.sqrt(self.d)
        out = self.pos_embedding(out)
        x = self.block1(out, src_mask)
        return self.layer_stack(x)

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, d1, d2, N):
        super().__init__()
        self.d2 = d2
        self.qL = nn.Linear(
            in_features = d2,
            out_features = d2
        )
        self.kL = nn.Linear(
            in_features = d1,
            out_features = d2
        )
        self.vL = nn.Linear(
            in_features = d1,
            out_features = d2
        )
    
    def forward(self, x, y):
        Q = self.qL(y)
        K = self.kL(x)
        V = self.vL(x)

        energy = torch.matmul(Q, K.transpose(1, 2))
        energy /= math.sqrt(self.d2)
        attn = torch.softmax(energy, -1)

        return torch.matmul(attn, V)


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d1, d2, heads, num_words):
        super().__init__()
        self.mha = CustomMultiheadAttention(d2, heads)
        self.norm1 = nn.LayerNorm(d2)
        self.ca = CrossAttention(d1, d2, num_words)
        self.norm2 = nn.LayerNorm(d2)
        self.ff = nn.Sequential(
            nn.Linear(
                in_features = d2,
                out_features = d2*2
            ),
            nn.Tanh(),
            nn.Linear(
                in_features = d2*2,
                out_features = d2
            )
        )
        self.norm3 = nn.LayerNorm(d2)
    
    def forward(self, x, trg, mask = None):
        batch_size = x.shape[0]
        self_attn = self.mha(trg, mask, True)
        self_attn = self.norm1(self_attn + trg)
        cross_attn = self.ca(x, self_attn)
        cross_attn = self.norm2(cross_attn + self_attn)
        after_ff = self.ff(cross_attn)
        after_ff = self.norm3(after_ff + cross_attn)

        return after_ff

In [ ]:
class Decoder(nn.Module):
    def __init__(self, d1, d2, heads, num_layers, w2v_fr_root, max_len):
        super().__init__()
        self.embedding_fr = nn.Embedding.from_pretrained(torch.load(w2v_fr_root, weights_only=True),freeze=True).to(DEVICE)
        self.max_len = max_len
        self.d1 = d1
        self.d2 = d2
        self.heads = heads
        self.pos_embedding = PositionalEncoding(d2, max_len).to(DEVICE)
        self.pad_idx = 0
        self.layer_stack = nn.ModuleList([DecoderBlock(d1, d2, heads, max_len) for _ in range(num_layers-1)])
        self.block1 = DecoderBlock(self.d1, self.d2, self.heads, self.max_len).to(DEVICE)
    def forward(self, x, y):
        src_mask = (y != self.pad_idx).unsqueeze(1).unsqueeze(2).to(DEVICE)
        embeds = self.embedding_fr(y)
        embeds *= math.sqrt(self.d2)
        embeds = self.pos_embedding(embeds)

        out = self.block1(x, embeds, src_mask).to(DEVICE)
        
        for layer in self.layer_stack:
            out = layer(x, out)
        return out

In [ ]:
class CustomTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(50, 5, 2, "weights/eng_vectors.pt", 40)
        self.decoder = Decoder(50, 50, 5, 2, "weights/fr_vectors.pt", 40)
        self.linear = nn.Linear(
            in_features = 50,
            out_features = len(index2word_fr)
        )
    
    def forward(self, x, y):
        x = self.encoder(x)
        out = self.decoder(x, y)
        return self.linear(out)

In [ ]:
model = CustomTransformer()
model = model.to(DEVICE)

In [ ]:
def trainer(model, dataloader, EPOCHS, pad_idx=0):
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4) 
    
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx)
    
    for epoch in range(EPOCHS):
        model.train() 
        
        epoch_loss = 0
        total_correct_epoch = 0
        total_words_epoch = 0
        
        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True) 
        
        for eng, fr in loop:
            eng = eng.to(DEVICE)
            fr = fr.to(DEVICE)
            
            optimizer.zero_grad()
            
            # The Shift (Perfectly aligned)
            decoder_input = fr[:, :-1] 
            targets = fr[:, 1:] 
            
            preds = model(eng, decoder_input)
            
            preds_flat = preds.reshape(-1, preds.shape[-1])
            targets_flat = targets.reshape(-1)
            
            loss = loss_fn(preds_flat, targets_flat)
            loss.backward()
            
            # UPGRADE 3: Gradient Clipping (Prevents loss from exploding to NaN)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            epoch_loss += loss.item()

            # --- True Accuracy Tracking ---
            pred_labels = torch.argmax(preds, dim=-1)
            non_pad_elements = (targets != pad_idx)
            
            batch_correct = ((pred_labels == targets) & non_pad_elements).sum().item()
            batch_words = non_pad_elements.sum().item()
            
            total_correct_epoch += batch_correct
            total_words_epoch += batch_words
            
            current_batch_acc = (batch_correct / batch_words) * 100 if batch_words > 0 else 0
            loop.set_postfix(loss=loss.item(), acc=current_batch_acc)
            
        avg_loss = epoch_loss / len(dataloader)
        avg_acc = (total_correct_epoch / total_words_epoch) * 100 if total_words_epoch > 0 else 0

        print(f"\nEpoch {epoch+1} Finished | Avg Loss = {avg_loss:.5f} | True Accuracy = {avg_acc:.3f}%")
        torch.save(model.state_dict(), 'weights/transformer_model.pt')

In [ ]:
from torchinfo import summary

with torch.inference_mode():
    eng, fr = next(iter(dataloader))
    eng, fr = eng.to(DEVICE), fr.to(DEVICE)
    model2 = CustomTransformer()
    model2.to(DEVICE)
    
    print(summary(
        model2, 
        input_data=[eng[:, 1:], fr[:, :-1]],
    ))

Layer (type:depth-idx)                                  Output Shape              Param #
CustomTransformer                                       [32, 39, 42455]           --
├─Encoder: 1-1                                          [32, 39, 50]              --
│    └─Embedding: 2-1                                   [32, 39, 50]              (836,000)
│    └─PositionalEncoding: 2-2                          [32, 39, 50]              --
│    │    └─Dropout: 3-1                                [32, 39, 50]              --
│    └─EncoderBlock: 2-3                                [32, 39, 50]              --
│    │    └─CustomMultiheadAttention: 3-2               [32, 39, 50]              10,200
│    │    └─LayerNorm: 3-3                              [32, 39, 50]              100
│    │    └─Sequential: 3-4                             [32, 39, 50]              10,150
│    │    └─LayerNorm: 3-5                              [32, 39, 50]              100
│    └─Sequential: 2-4                     

In [ ]:
trainer(model, dataloader, 2)

Epoch 1/2:   0%|          | 0/7516 [00:00<?, ?it/s]


RuntimeError: The size of tensor a (40) must match the size of tensor b (39) at non-singleton dimension 1